# P-2026-11-24-1: Why does the lab's grouped ECI rank Nigeria 20th and Congo 43rd of 105 (2019)?

**Question** (from `proposals/2026-11-21.md` #2, in open PR #17): on the joint common sample used by 11-19 to 11-21, the lab's 18-group ECI ranks **Nigeria 20th** and the **Republic of Congo 43rd** of 105 in 2019. The Growth Lab's published ECI (ATLAS) ranks them 104th and 101st. What in their 18-group rows produces this? Is it a mapping or data artefact, does it affect other countries, and is there a cheap fix that also helps the rest of the sample?

**Why tonight:** `QUEUE.md` Q01–Q16 are all taken on `main`. Every open `claude/nightly-*` branch (PRs #16–#19, 11-20 to 11-23) already has a run that took the top idea from the previous night's proposals. Under PROTOCOL §1.4, tonight's question comes from `proposals/`. 11-21 #1 ("resource-adjusted expected income") is about report interpretation. 11-21 #3 was taken by 11-22, and 11-22 #1 by 11-23. 11-21 #2 is the highest-value untaken idea that is about the ECI *as the lab computes it*. Every report uses the grouped ECI, and 11-21 flagged this anomaly as something to understand before any further grouped tweaks.

- **Date:** 2026-11-24 (America/New_York), nightly run. ID `P-2026-11-24-1`.
- **Data:** `data/cleaned/2019–2023clean.csv` (grouped, via `lib/research.lab_groups_iso3`, EU-28 dropped); `data/raw/2019–2023_mega.csv` (HS6: drill-down, HS4 base ECI, resource share RS = ch. 26/27/71, and the rebuilt groupings in E); `external-data/raw/atlas_hs92_country_year.csv` (ATLAS ECI, benchmark only); `external-data/raw/atlas_hs92_product_year_4.csv` (ATLAS HS4 PCI, benchmark only); `external-data/raw/wdi_panel.csv` (GDPpc PPP). No downloads.
- **Years:** 2019 primary; 2020–2022 as checks; 2023 shown but suspect (D1).
- **Library:** `lib/eci.py` and `lib/research.py`, imported unchanged. O4 orientation is copied from 11-19/11-21.
- **Run time:** about 13 s (printed in the last cell).

## TL;DR

- **It is a mapping artefact, not a capability signal.** Grouped ECI is *exactly* the mean PCI of a country's RCA>1 groups (identity checked to 4e-16). Nigeria's basket is {Energy, Machinery and Construction, Daily Instruments}. Two of those are the **3rd- and 4th-most complex of the 18 groups**, and each comes from a few HS6 lines unrelated to manufacturing capability. **Daily Instruments** = $3.18B, 71% of it one line, **890800 (vessels for breaking up, $2.256B)**, plus 890520 (floating drilling/production platforms, $0.550B). **Machinery and Construction** = $2.097B, of which **830710 (flexible steel tubing) is $2.096B**, a line that appears only in 2019. Congo's case is the same Daily Instruments effect (ch. 89 vessels and platforms, $0.286B).
- **Removing those lines from their own rows brings both back to their ATLAS rank.** Nigeria: 20 → **102/105** (both removed; 43 with only ch. 89 removed, 42 with only 830710 removed). Congo: 43 → **86/105** (ch. 89 removed). ATLAS: 104 and 101. In the Growth Lab's own product space these goods are *low*-complexity: 2019 ATLAS PCI percentile **0.113** (8905) and **0.115** (8908). The lab mapping puts HS ch. 89 (ships) in *Daily Instruments* rather than *Transportation*.
- **The pattern is general but small.** 6 countries in 2019 rank ≥40 places above ATLAS on grouped ECI (NGA, COD, COG, ZMB, MDA, SAU). **All 6** have a "thin" high-PCI cell (one HS6 line ≥50% of the cell), vs **41.4%** of the other 99 (Fisher p = 0.0067). Examples: DR Congo's cobalt hydroxide (282200) in *Chemicals*; Saudi Arabia's unspecified 999999, which feeds both *High-complexity Products* and *Services and Utilities*.
- **Moving ch. 89 to Transportation (F1) passes the pre-stated rule A** (beats the rebuilt base on both ATLAS agreement and income fit in 2019 and in 3 of 2020–2022). 2019: ΔM1 **+0.037** [−0.002, 0.095], ΔM2 **+0.034** [0.004, 0.077]. The full Q05 Table-1 mapping (F2) also passes (ΔM1 +0.060, ΔM2 +0.038; both CIs include 0). **But the gain is almost all Nigeria and Congo.** Excluding those two, F1's ΔM1 is +0.003 / −0.008 / −0.006 / +0.001 (2019–2022). So F1 is a targeted fix with few side effects: 4 of 105 countries move ≥10 ranks in 2019, including Poland 33 → 23. It is not a broad improvement.
- **The generic "thin-cell guard" (F3) is rejected.** It removes 221 of 736 M=1 cells and drops 5 countries entirely. It also makes the **UAE #1** (2019 and 2022), so it fails rule A.

## Why this matters for the lab

Every country report prints a grouped-ECI rank. If a country can jump 80 places because of one shipping line, two things break. The report's ECI ranks don't measure capabilities for those countries, and the greedy optimizer's target (the marginal ECI gain from boosting a group) is set partly by misfiled products. Agenda §0, link 3, needs the group PCIs to mean what their names say. This run tests whether the worst grouped/ATLAS disagreements are economics or plumbing. If they are plumbing, it identifies the smallest change to the cleaning mapping (`code/data-retrieval/datacleaning2025.ipynb`, which Will owns) that removes them without rewriting everyone else's rank.

## Method (pre-stated before any variant was scored)

**Base:** the lab's grouped ECI (drop-any-zero filter, strict `RCA > 1`, 18 columns, sorted SVD, O4 orientation). Ranks are within the **joint common sample** of 11-21 (grouped base ∩ HS4 base ∩ ATLAS ∩ WDI ∩ RS: 105 countries in 2019).

**Diagnosis (A–C):**
1. Verify the basket-mean identity `k_c = mean_{p∈basket} q_p / σ₂` (it follows from the SVD). Then ECI rank is fully determined by which groups a country has RCA>1 in, and those groups' PCI.
2. Drill down from each of Nigeria's and Congo's RCA>1 cells to the HS6 lines in `data/raw`.
3. Exact counterfactuals on **their own rows only**: remove the ch. 89 lines and/or HS6 830710, recompute the whole ECI, and report the rank.

**Generality (D), pre-stated:** *anomalous* = grouped rank at least 40 places better than ATLAS rank (same year, common sample). *Thin cell* = an RCA>1 cell where the single largest HS6 line is ≥50% of the country's exports in that group. *High-PCI* = group PCI > 0. Test: share of anomalous vs other countries with ≥1 thin high-PCI cell (Fisher exact, each year 2019–2022).

**Lab-wide variants (E)**, all rebuilt from `data/raw` with `research.groups_from_raw`, so they are compared with **R0** = the lab mapping rebuilt the same way (R0 vs the cleaned CSV: Spearman 0.987–1.000):
- **F1** ch. 89 moved from *Daily Instruments* to *Transportation* (HS section XVII, "vessels", sits with 86–88).
- **F2** the Q05 Table-1 mapping (84+85 → Machinery, 86–89 → Transportation, 83 → Metals, 96/97 → Daily Instruments). It includes F1.
- **F3** thin-cell guard on R0: set M = 0 for every thin cell.

**Metrics and rule** (as 11-19 to 11-21): **M1** = Spearman with ATLAS ECI, **M2** = Spearman with log GDPpc (PPP). **Rule A:** a variant is *better* only if it beats R0 on both M1 and M2 in 2019 **and** in ≥2 of 2020–2022. 2019 paired country bootstrap (2,000 draws). Each variant is compared with R0 on the same countries.

*Post hoc (labelled as such):* E4 repeats the M1/M2 differences with Nigeria and Congo excluded, and with all six 2019 anomalous countries excluded.

## Setup and reproduction of 11-21's ranks

In [1]:
import sys, time, copy, warnings
T0 = time.time()
sys.path.insert(0, '../../lib')
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from scipy import stats
import eci, research as R
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 30); pd.set_option('display.precision', 3)
FIG = Path('figures'); FIG.mkdir(exist_ok=True)
YEARS = [2019, 2020, 2021, 2022, 2023]; CHK = [2019, 2020, 2021, 2022]
FOCUS = ['NGA', 'COG']
M49 = {'NGA': 566, 'COG': 178}
RES_CH = ['26', '27', '71']

def o4(M):
    """Sorted SVD of binary M, O4 orientation (PCI negative with ubiquity). Copied from 11-19/11-21.
    Returns z-scored ECI, raw k (ECI before z-scoring), raw q (PCI), s2 (2nd singular value)."""
    Mv = M.to_numpy(float); kc, kp = Mv.sum(1), Mv.sum(0)
    B = Mv / np.sqrt(kc[:, None] * kp[None, :])
    U, S, Vt = np.linalg.svd(B, full_matrices=False)
    k = U[:, 1] / np.sqrt(kc); q = Vt[1] / np.sqrt(kp)
    s4 = -np.sign(np.corrcoef(q, kp)[0, 1]); k, q = k * s4, q * s4
    e = pd.Series(k, index=M.index); z = (e - e.mean()) / e.std()
    return z, e, pd.Series(q, index=M.columns), S[1]

def lab_M(X):
    """Lab rules: drop any country with a zero group, strict RCA > 1."""
    X = eci.notebook_filter(X)
    Mm = (eci.rca(X) > 1).astype(float)
    return Mm.loc[Mm.sum(1) > 0, Mm.sum(0) > 0]

def hs_M(X):
    Mm = (eci.rca(X) >= 1).astype(float)
    return Mm.loc[Mm.sum(1) > 0, Mm.sum(0) > 0]

def rank(e, idx):
    return e[idx].rank(ascending=False)

G = {y: R.lab_groups_iso3(y, drop_eu=True) for y in YEARS}
XH = {y: R.repo_hs(y, 4) for y in YEARS}
RS = {y: XH[y][[c for c in XH[y].columns if c[:2] in RES_CH]].sum(1) / XH[y].sum(1) for y in YEARS}
AT = R.atlas_cy(); ATL = {y: AT.query('year == @y').set_index('iso3').eci for y in YEARS}
LNY = {y: np.log(R.gdppc(y, 'ppp')).dropna() for y in YEARS}
GB, HB = {}, {}
for y in YEARS:
    GB[y] = o4(lab_M(G[y])); HB[y] = o4(hs_M(XH[y]))
COMMON = {y: sorted(set(GB[y][0].index) & set(HB[y][0].index) & set(ATL[y].dropna().index) & set(LNY[y].index) & set(RS[y].index)) for y in YEARS}
print('grouped base n:', {y: len(GB[y][0]) for y in YEARS}, '| joint common sample n:', {y: len(COMMON[y]) for y in YEARS})
rows = []
for y in YEARS:
    c = COMMON[y]
    for f in FOCUS:
        if f in c:
            rows.append({'year': y, 'iso3': f, 'n': len(c), 'grouped rank': int(rank(GB[y][0], c)[f]), 'HS4 rank': int(rank(HB[y][0], c)[f]),
                         'ATLAS rank': int(rank(ATL[y], c)[f]), 'lnGDPpc rank': int(rank(LNY[y], c)[f]), 'RS': round(RS[y][f], 2)})
        else:
            rows.append({'year': y, 'iso3': f, 'n': len(c), 'grouped rank': None, 'note': 'not in common sample' + ('' if f in GB[y][0].index else ' (not in grouped base)')})
print(pd.DataFrame(rows).to_string(index=False))

grouped base n: {2019: 120, 2020: 120, 2021: 105, 2022: 120, 2023: 122} | joint common sample n: {2019: 105, 2020: 105, 2021: 94, 2022: 105, 2023: 109}
 year iso3   n  grouped rank  HS4 rank  ATLAS rank  lnGDPpc rank   RS                                       note
 2019  NGA 105          20.0     105.0       104.0          87.0 0.87                                        NaN
 2019  COG 105          43.0      84.0       101.0          89.0 0.84                                        NaN
 2020  NGA 105          55.0     103.0       104.0          87.0 0.89                                        NaN
 2020  COG 105          47.0      86.0       103.0          90.0 0.75                                        NaN
 2021  NGA  94          94.0      94.0        94.0          82.0 0.89                                        NaN
 2021  COG  94          50.0      73.0        91.0          83.0 0.55                                        NaN
 2022  NGA 105         103.0     103.0       105.0       

**Takeaway:** the 11-21 numbers reproduce exactly. 2019: Nigeria 20/105 grouped vs 104 ATLAS (HS4 105); Congo 43/105 vs 101 (HS4 84). Nigeria's anomaly fades after 2019 (55 in 2020, 94/94 in 2021, 103 in 2022). Congo stays high in 2020 (47) and 2021 (50/94), and is absent from the 2022–2023 raw pull.

## Results

### A. Grouped PCI and the two baskets (2019)

In [2]:
y = 2019
M0 = lab_M(G[y]); z0, k0, q0, s0 = GB[y]
# basket-mean identity: k_c = (1/s) * mean_{p in basket} q_p
pred = (M0.to_numpy() @ q0.to_numpy()) / M0.sum(1).to_numpy() / s0
print(f'basket-mean identity, max |k - mean(q)/s| over {len(k0)} countries: {np.abs(pred - k0.to_numpy()).max():.2e}')
pci = pd.DataFrame({'PCI (raw q)': q0, 'ubiquity': M0.sum(0).astype(int)}).sort_values('PCI (raw q)', ascending=False)
pci['PCI rank (of 18)'] = range(1, len(pci) + 1)
print(pci.to_string())
c = COMMON[y]
tab = []
for f in FOCUS + ['AGO', 'AZE', 'KWT', 'SAU', 'DZA', 'GAB', 'GHA']:
    if f not in M0.index: continue
    b = list(M0.columns[M0.loc[f] == 1])
    tab.append({'iso3': f, 'RS': round(RS[y].get(f, np.nan), 2), 'diversity': len(b), 'basket mean PCI': round(q0[b].mean(), 4),
                'grouped rank /105': int(rank(z0, c)[f]) if f in c else None, 'ATLAS rank /105': int(rank(ATL[y], c)[f]) if f in c else None,
                'basket (RCA)': ', '.join(f'{g} ({eci.rca(eci.notebook_filter(G[y])).loc[f, g]:.2f})' for g in b)})
print(pd.DataFrame(tab).to_string(index=False))

basket-mean identity, max |k - mean(q)/s| over 120 countries: 3.96e-16
                               PCI (raw q)  ubiquity  PCI rank (of 18)
Transportation                   8.641e-02        20                 1
High-complexity Products         8.012e-02        24                 2
Daily Instruments                5.395e-02        35                 3
Machinery and Construction       5.352e-02        24                 4
Raw Manufacturing                3.717e-02        35                 5
Chemicals                        3.224e-02        29                 6
Chemical Manufacturing           2.538e-02        39                 7
Services and Utilities           1.932e-02        19                 8
Weapons                          1.772e-02        26                 9
Clothing and Accessories        -3.511e-04        39                10
Wood Products                   -1.106e-03        43                11
Metals                          -2.229e-03        58                12
Textil

**Takeaway (A):** the basket-mean identity holds to 3.96e-16, so a rank is set entirely by the basket. *Daily Instruments* (PCI rank 3 of 18) and *Machinery and Construction* (rank 4) are among the most "complex" groups. Nigeria has RCA 1.07 and 11.44 in them, which lifts its basket mean to 0.021, against −0.050 for Angola, whose basket is Energy + Precious Stones and ranks 104. Congo gets its lift (basket mean 0.003) from *Daily Instruments* (RCA 1.01). Its other non-oil cell, *Wood Products*, sits mid-table (PCI rank 11). Saudi Arabia ranks **2/105** vs ATLAS 45 on a basket of Chemicals, High-complexity Products and Services and Utilities (RCA 10.69). D below explains why.

In [3]:
fig, ax = plt.subplots(figsize=(9, 4.4))
order = pci.index[::-1]
ax.barh(range(len(order)), pci.loc[order, 'PCI (raw q)'], color='#d6d5d1')
for f, col, dy in [('NGA', '#d1495b', 0.18), ('COG', '#2a78d6', -0.18)]:
    b = M0.columns[M0.loc[f] == 1]
    for g in b:
        i = list(order).index(g); ax.scatter(pci.loc[g, 'PCI (raw q)'], i + dy, color=col, s=36, zorder=3, label=f if g == b[0] else None)
    ax.axvline(q0[b].mean(), color=col, ls='--', lw=1.2)
ax.axvline(q0[M0.columns[M0.loc['AGO'] == 1]].mean(), color='#52514e', ls=':', lw=1.2, label='AGO basket mean')
ax.set_yticks(range(len(order))); ax.set_yticklabels(order, fontsize=8)
ax.set_xlabel('grouped PCI (raw eigenvector, O4), 2019'); ax.legend(frameon=False, fontsize=8, loc='lower right')
ax.set_title('Dots = groups with RCA > 1; dashed = basket mean PCI (sets ECI exactly)', fontsize=9, loc='left')
for s_ in ['top', 'right']: ax.spines[s_].set_visible(False)
fig.tight_layout(); fig.savefig(FIG / 'baskets_2019.png', dpi=100); plt.close(fig)
print('saved figures/baskets_2019.png')

saved figures/baskets_2019.png


![baskets](figures/baskets_2019.png)

**Takeaway (figure):** both countries' basket means (dashed) sit far to the right of Angola's (dotted), and the gap comes from their dots in the top-4 groups.

### B. What is inside those cells? (HS6 drill-down)

In [4]:
HS_TITLES = {'890800': 'vessels and other floating structures for breaking up', '890520': 'floating or submersible drilling or production platforms',
             '830710': 'flexible tubing of iron or steel', '890190': 'other vessels for transport of goods / persons', '890590': 'other special-purpose vessels (light-vessels, dredgers, floating cranes...)',
             '890110': 'cruise ships, ferries', '890400': 'tugs and pusher craft', '890510': 'dredgers'}
def ch_of(g): return R.LAB_GROUPS[g]
rows = []
for f in FOCUS:
    raw = R.raw_hs6(y)[M49[f]]
    for g in M0.columns[M0.loc[f] == 1]:
        s = raw[raw.index.str[:2].astype(int).isin(ch_of(g))]; s = s[s > 0].sort_values(ascending=False)
        top = s.head(3)
        rows.append({'iso3': f, 'group': g, 'cleaned $B': round(G[y].loc[f, g] / 1e9, 3), 'raw HS6 sum $B': round(s.sum() / 1e9, 3),
                     'n HS6 lines': len(s), 'top HS6 share': round(top.iloc[0] / s.sum(), 3),
                     'top lines ($B)': '; '.join(f'{k} {v/1e9:.3f}' for k, v in top.items())})
print(pd.DataFrame(rows).to_string(index=False))
print('\nHS titles used for labels (HS nomenclature, not repo data):'); [print(' ', k, v) for k, v in HS_TITLES.items()]

iso3                      group  cleaned $B  raw HS6 sum $B  n HS6 lines  top HS6 share                            top lines ($B)
 NGA Energy Drilling and Mining      46.737          46.737           42          0.878 270900 41.040; 271111 4.980; 271129 0.315
 NGA Machinery and Construction       2.097           2.097            2          1.000                830710 2.096; 830990 0.001
 NGA          Daily Instruments       3.183           3.183           23          0.709  890800 2.256; 890520 0.550; 890190 0.197
 COG Energy Drilling and Mining       4.690           4.690           33          0.940  270900 4.409; 271019 0.207; 271012 0.036
 COG              Wood Products       0.333           0.333           37          0.574  440349 0.191; 440727 0.076; 440729 0.046
 COG          Daily Instruments       0.302           0.302           96          0.506  890190 0.153; 890520 0.088; 890590 0.041

HS titles used for labels (HS nomenclature, not repo data):
  890800 vessels and other fl

[None, None, None, None, None, None, None, None]

**Takeaway (B):** both suspect cells come from a few lines. Nigeria's *Machinery and Construction* is 2 HS6 lines, and **830710 is $2.096B of $2.097B** (share 1.000). Its *Daily Instruments* is mostly ch. 89: **890800 $2.256B** (top share 0.709), **890520 $0.550B**, 890190 $0.197B. Congo's *Daily Instruments* top lines are also ch. 89 (890190 $0.153B, 890520 $0.088B, 890590 $0.041B). The cleaned cells equal the raw HS6 sums for both countries, so the cleaning step adds no error here. The mapping is the cause: the lab puts ch. 89 in *Daily Instruments* (with 90–95: instruments, clocks, furniture, toys).

In [5]:
rows = []
for yy in YEARS:
    raw = R.raw_hs6(yy)
    for f in FOCUS:
        if M49[f] not in raw.columns: rows.append({'year': yy, 'iso3': f, 'note': 'reporter absent from raw pull'}); continue
        s = raw[M49[f]]; tot = s.sum()
        r = {'year': yy, 'iso3': f, 'total $B': round(tot / 1e9, 2), 'ch.89 $B': round(s[s.index.str[:2] == '89'].sum() / 1e9, 3),
             '830710 $B': round(s.get('830710', 0) / 1e9, 3)}
        if f in G[yy].index:
            Rm = eci.rca(G[yy]); r['RCA Daily Instr.'] = round(Rm.loc[f, 'Daily Instruments'], 2); r['RCA Mach.&Constr.'] = round(Rm.loc[f, 'Machinery and Construction'], 2)
        r['in grouped base'] = f in GB[yy][0].index
        r['grouped rank'] = f"{int(rank(GB[yy][0], COMMON[yy])[f])}/{len(COMMON[yy])}" if f in COMMON[yy] else '-'
        rows.append(r)
print(pd.DataFrame(rows).to_string(index=False))

 year iso3  total $B  ch.89 $B  830710 $B  RCA Daily Instr.  RCA Mach.&Constr. in grouped base grouped rank                          note
 2019  NGA     53.62     3.181      2.096              1.08              11.58            True       20/105                           NaN
 2019  COG      5.58     0.286      0.000              1.02               0.01            True       43/105                           NaN
 2020  NGA     34.90     2.173      0.000              1.12               0.01            True       55/105                           NaN
 2020  COG      4.89     0.699      0.000              2.62               0.06            True       47/105                           NaN
 2021  NGA     47.23     1.437      0.000              0.57               0.01            True        94/94                           NaN
 2021  COG      2.36     0.337      0.000              2.62               0.00            True        50/94                           NaN
 2022  NGA     63.42     0.775    

**Takeaway (B2):** **830710 is a 2019-only line for Nigeria** ($2.096B in 2019, $0.000B in 2020–2023). Nigeria's RCA in *Machinery and Construction* falls from 11.58 to 0.01 the next year. That is the signature of a one-off reporting or classification event, not a capability (see the data request). Nigeria's ch. 89 exports shrink every year ($3.181B → $0.198B, 2019 → 2023), and its *Daily Instruments* RCA drops below 1 from 2021. This is why its grouped rank converges to ATLAS by 2021. Congo's *Daily Instruments* RCA is 2.62 in 2020 and 2021, so its anomaly persists.

In [6]:
AP = pd.read_csv(R.EXT / 'atlas_hs92_product_year_4.csv', dtype={'product_hs92_code': str}).query('year == 2019')
AP = AP.assign(pci_pct=AP.pci.rank(pct=True)).set_index('product_hs92_code')
print(f'ATLAS 2019 PCI (n = {AP.pci.notna().sum()} HS4 products; pct = percentile, 1 = most complex):')
print(AP.loc[['8307', '8905', '8908', '8901', '2709', '4403', '8703', '9018'], ['pci', 'pci_pct']].round(3).to_string())

ATLAS 2019 PCI (n = 1243 HS4 products; pct = percentile, 1 = most complex):
                     pci  pci_pct
product_hs92_code                
8307               0.499    0.636
8905              -1.297    0.113
8908              -1.293    0.115
8901              -0.678    0.244
2709              -3.035    0.004
4403              -1.676    0.067
8703               0.969    0.843
9018               0.563    0.665


**Takeaway (B3):** in the Growth Lab's own 2019 product space, the ch. 89 goods behind both anomalies are **low**-complexity. 8905 (platforms, dredgers) is at the **0.113** percentile and 8908 (vessels for breaking up) at **0.115**. Both are close to crude logs (4403, 0.067) and far from cars (8703, 0.843). 8307 (flexible metal tubing) is mid-high (0.636). The lab's grouping assigns all of these the high PCI of their group, which is how scrap ships end up raising an oil exporter's ECI.

### C. Exact counterfactuals on the two countries' own rows (2019)

In [7]:
def without_lines(X, yy, f, chapters=(), hs6=()):
    """Remove the given HS6 lines / chapters from country f's cleaned group values (all groups the chapter feeds)."""
    X = X.copy(); raw = R.raw_hs6(yy)[M49[f]]
    sel = raw[(raw.index.str[:2].isin(list(chapters))) | (raw.index.isin(list(hs6)))]
    for code, v in sel.items():
        ch = int(code[:2])
        for g, chs in R.LAB_GROUPS.items():
            if ch in chs and g in X.columns:
                X.loc[f, g] = max(X.loc[f, g] - v, 0.0)
    return X
CF = {'CF0 as is': {}, 'CF1 drop ch.89 lines': {'chapters': ['89']}, 'CF2 drop 830710': {'hs6': ['830710']},
      'CF3 drop both': {'chapters': ['89'], 'hs6': ['830710']}}
rows = []
for f in FOCUS:
    for name, kw in CF.items():
        X = without_lines(G[y], y, f, **kw) if kw else G[y]
        Mx = lab_M(X); z = o4(Mx)[0]
        cc = [i for i in COMMON[y] if i in z.index]
        rows.append({'iso3': f, 'counterfactual (own data only)': name, 'in sample': f in z.index,
                     'basket': ', '.join(Mx.columns[Mx.loc[f] == 1]) if f in Mx.index else '-',
                     'rank': f"{int(rank(z, cc)[f])}/{len(cc)}" if f in cc else '-', 'ATLAS rank': int(rank(ATL[y], COMMON[y])[f])})
print(pd.DataFrame(rows).to_string(index=False))

iso3 counterfactual (own data only)  in sample                                                                    basket    rank  ATLAS rank
 NGA                      CF0 as is       True Energy Drilling and Mining, Machinery and Construction, Daily Instruments  20/105         104
 NGA           CF1 drop ch.89 lines       True                    Energy Drilling and Mining, Machinery and Construction  43/105         104
 NGA                CF2 drop 830710       True                             Energy Drilling and Mining, Daily Instruments  42/105         104
 NGA                  CF3 drop both       True                                                Energy Drilling and Mining 102/105         104
 COG                      CF0 as is       True              Energy Drilling and Mining, Wood Products, Daily Instruments  43/105         101
 COG           CF1 drop ch.89 lines       True                                 Energy Drilling and Mining, Wood Products  86/105         101
 COG         

**Takeaway (C):** each suspect cell alone is worth about 22 ranks for Nigeria (20 → 43 without ch. 89, 20 → 42 without 830710). Removing both leaves the basket {Energy} and Nigeria ranks **102/105**, next to ATLAS (104). For Congo, removing ch. 89 loses the *Daily Instruments* cell and moves it **43 → 86/105** (ATLAS 101). 830710 doesn't matter for Congo (rank unchanged at 43). The remaining Congo gap (86 vs 101) is its *Wood Products* cell (PCI rank 11), which looks like a genuine export (tropical logs: 440349 is 57.4% of the cell).

### D. Is this a two-country problem? Anomaly screen, 2019–2022

In [8]:
rows = []
for yy in CHK:
    c = COMMON[yy]; M_ = lab_M(G[yy]); q_ = GB[yy][2]
    gap = rank(GB[yy][0], c) - rank(ATL[yy], c)
    raw = R.raw_hs6(yy)
    iso2m = {}
    for code in raw.columns:
        i3 = eci.m49_to_iso3(code)
        if i3: iso2m.setdefault(i3, []).append(code)
    for f in c:
        b = M_.columns[M_.loc[f] == 1]
        cols = [m for m in iso2m.get(f, []) if m not in eci.M49_AGGREGATES]
        s_all = raw[cols].sum(1) if cols else None
        thin = []
        for g in b:
            if s_all is None: break
            s = s_all[s_all.index.str[:2].astype(int).isin(R.LAB_GROUPS[g])]
            if s.sum() > 0 and s.max() / s.sum() >= 0.5: thin.append(g)
        rows.append({'year': yy, 'iso3': f, 'gap': gap[f], 'anomalous': gap[f] <= -40, 'diversity': len(b), 'thin': len(thin),
                     'thin high-PCI': sum(q_[g] > 0 for g in thin), 'thin groups': ', '.join(thin), 'RS': RS[yy][f]})
SC = pd.DataFrame(rows)
summ = SC.groupby(['year', 'anomalous']).agg(n=('iso3', 'size'), mean_div=('diversity', 'mean'), share_with_thin=('thin', lambda s: (s > 0).mean()),
                                             share_with_thin_highPCI=('thin high-PCI', lambda s: (s > 0).mean()), mean_RS=('RS', 'mean'))
print(summ.round(3).to_string())
print('\nAnomalous countries (grouped rank >= 40 places better than ATLAS), 2019:')
print(SC.query('year == 2019 and anomalous').sort_values('gap')[['iso3', 'gap', 'diversity', 'thin', 'thin high-PCI', 'thin groups', 'RS']].round(2).to_string(index=False))
for yy in CHK:
    a = SC.query('year == @yy'); t = pd.crosstab(a.anomalous, a['thin high-PCI'] > 0)
    p = stats.fisher_exact(t.to_numpy())[1] if t.shape == (2, 2) else np.nan
    print(yy, 'Fisher exact p (anomalous vs any thin high-PCI cell):', round(p, 4), '| anomalous n =', int(a.anomalous.sum()))
print('countries anomalous in >= 3 of 2019-2022:', SC.groupby('iso3').anomalous.sum().loc[lambda s: s >= 3].to_dict())

# top HS6 line in each thin cell of the 2019 anomalous countries
raw = R.raw_hs6(2019); rows = []
for f in SC.query('year == 2019 and anomalous').sort_values('gap').iso3:
    cols = [m for m in raw.columns if eci.m49_to_iso3(m) == f and m not in eci.M49_AGGREGATES]; sa = raw[cols].sum(1)
    for g in SC.query('year == 2019 and iso3 == @f')['thin groups'].iloc[0].split(', '):
        sg = sa[sa.index.str[:2].astype(int).isin(R.LAB_GROUPS[g])]
        rows.append({'iso3': f, 'thin group': g, 'group PCI rank (of 18)': int(pci.loc[g, 'PCI rank (of 18)']), 'top HS6': sg.idxmax(),
                     'top HS6 $B': round(sg.max() / 1e9, 3), 'share of cell': round(sg.max() / sg.sum(), 3)})
print(pd.DataFrame(rows).to_string(index=False))

                  n  mean_div  share_with_thin  share_with_thin_highPCI  mean_RS
year anomalous                                                                  
2019 False       99     6.192            0.798                    0.414    0.267
     True         6     3.333            1.000                    1.000    0.319
2020 False      101     6.168            0.792                    0.347    0.246
     True         4     2.750            1.000                    1.000    0.446
2021 False       92     6.120            0.793                    0.478    0.253
     True         2     4.000            1.000                    0.500    0.415
2022 False      101     6.059            0.812                    0.455    0.283
     True         4     4.250            1.000                    1.000    0.358

Anomalous countries (grouped rank >= 40 places better than ATLAS), 2019:
iso3   gap  diversity  thin  thin high-PCI                                                               thin groups

**Takeaway (D):** in 2019, **6 of 105** countries rank ≥40 places above ATLAS. **All 6** have at least one thin high-PCI cell, vs **41.4%** of the other 99 (Fisher p = **0.0067**). The pattern holds in 2020 (4 anomalous, p = 0.0172) and 2022 (4, p = 0.0482), but not in 2021 (2 anomalous, p = 1.0). The thin lines are commodities or unspecified goods in "complex" groups:
- **DR Congo:** cobalt oxides/hydroxides **282200** ($3.922B, 99.6% of its *Chemicals* cell, PCI rank 6).
- **Zambia:** 280700 in *Chemicals* and 360200 (explosives) in *Weapons*.
- **Saudi Arabia:** the unspecified line **999999** ($202.382B, which 11-17 identified as mostly its oil). It feeds both *Services and Utilities* and the *High-complexity Products* aggregate, because ch. 99 is in that aggregate's chapter list.
- **Moldova:** 940190 (seat parts) in *Daily Instruments*.

Only Congo is anomalous in ≥3 of 2019–2022. Thin cells themselves are common: 79.8% of non-anomalous countries have one, mostly their main commodity line in a low-PCI group. That is why a blanket thin-cell rule (F3, below) goes too far.

### E. Lab-wide fixes vs the rebuilt lab mapping (R0)

In [9]:
L0 = R.LAB_GROUPS
F1 = copy.deepcopy(L0); F1['Daily Instruments'] = [c for c in L0['Daily Instruments'] if c != 89]; F1['Transportation'] = L0['Transportation'] + [89]
T1 = copy.deepcopy(L0); T1['Machinery and Construction'] = [84, 85]; T1['Transportation'] = [86, 87, 88, 89, 98]
T1['Metals'] = list(range(72, 84)); T1['Daily Instruments'] = [90, 91, 93, 94, 95, 96, 97]          # = Q05 'table1'
MAPS = {'R0 rebuilt lab mapping': L0, 'F1 ch.89 -> Transportation': F1, 'F2 Table-1 mapping (Q05)': T1}

VAR = {}
for yy in YEARS:
    for name, mp in MAPS.items():
        X = R.groups_from_raw(yy, mp)
        VAR[yy, name] = o4(lab_M(X))[0]
    # F3: thin-cell guard on M (cells whose top HS6 >= 50% are set to M = 0), R0 mapping
    X = R.groups_from_raw(yy, L0); M_ = lab_M(X)
    raw = R.raw_hs6(yy); raw = raw[[c for c in raw.columns if c not in eci.M49_AGGREGATES]]
    raw.columns = [eci.m49_to_iso3(c) for c in raw.columns]; raw = raw.loc[:, pd.notna(raw.columns)].T.groupby(level=0).sum().T
    ch = raw.index.str[:2].astype(int)
    flips = 0
    for g in M_.columns:
        sub = raw[np.isin(ch, L0[g])]; tot, mx = sub.sum(), sub.max()
        for i in M_.index[M_[g] == 1]:
            if tot.get(i, 0) > 0 and mx[i] / tot[i] >= 0.5: M_.loc[i, g] = 0; flips += 1
    M_ = M_.loc[M_.sum(1) > 0, M_.sum(0) > 0]
    VAR[yy, 'F3 thin-cell guard (R0)'] = o4(M_)[0]
    if yy == 2019: print('F3 2019: M cells flipped 1->0:', flips, 'of', int(lab_M(X).to_numpy().sum()), 'ones')
VN = list(MAPS) + ['F3 thin-cell guard (R0)']
# Pairwise samples: each variant is compared with R0 on (joint common sample) ∩ R0 ∩ variant.
# F3 can remove every M cell of a country (all thin), which drops that country, so its sample can be smaller.
CS = {(yy, v): sorted(set(COMMON[yy]) & set(VAR[yy, VN[0]].index) & set(VAR[yy, v].index)) for yy in YEARS for v in VN}
print('pairwise sample n:', pd.DataFrame({v: [len(CS[yy, v]) for yy in YEARS] for v in VN}, index=YEARS).T.to_string())
print('countries F3 removes from the 2019 common sample:', sorted(set(CS[2019, VN[0]]) - set(CS[2019, VN[3]])))
rows = []
for yy in YEARS:
    for v in VN:
        c = CS[yy, v]; e, e0 = VAR[yy, v][c], VAR[yy, VN[0]][c]
        m1, m2 = stats.spearmanr(e, ATL[yy][c])[0], stats.spearmanr(e, LNY[yy][c])[0]
        rows.append({'year': yy, 'variant': v, 'n': len(c), 'M1 rho ATLAS': m1, 'M2 rho lnGDPpc': m2,
                     'M1 diff': m1 - stats.spearmanr(e0, ATL[yy][c])[0], 'M2 diff': m2 - stats.spearmanr(e0, LNY[yy][c])[0],
                     'rho w/ R0': stats.spearmanr(e, e0)[0],
                     'NGA rank': f"{int(rank(VAR[yy, v], c)['NGA'])}/{len(c)}" if 'NGA' in c else '-',
                     'COG rank': f"{int(rank(VAR[yy, v], c)['COG'])}/{len(c)}" if 'COG' in c else '-'})
P = pd.DataFrame(rows)
print(P.round(3).to_string(index=False))
P.round(4).to_csv(FIG / 'primary_metrics.csv', index=False)
print('R0 vs cleaned-CSV base, Spearman on the joint common sample:', {yy: round(stats.spearmanr(VAR[yy, VN[0]][CS[yy, VN[0]]], GB[yy][0][CS[yy, VN[0]]])[0], 3) for yy in YEARS})

F3 2019: M cells flipped 1->0: 221 of 736 ones


pairwise sample n:                             2019  2020  2021  2022  2023
R0 rebuilt lab mapping       105   105    94   105   109
F1 ch.89 -> Transportation   105   105    94   105   109
F2 Table-1 mapping (Q05)     105   105    94   105   109
F3 thin-cell guard (R0)      100    97    86    92   101
countries F3 removes from the 2019 common sample: ['AGO', 'BWA', 'COD', 'COG', 'NGA']
 year                    variant   n  M1 rho ATLAS  M2 rho lnGDPpc  M1 diff  M2 diff  rho w/ R0 NGA rank COG rank
 2019     R0 rebuilt lab mapping 105         0.689           0.453    0.000    0.000      1.000   18/105   41/105
 2019 F1 ch.89 -> Transportation 105         0.726           0.487    0.037    0.034      0.983   39/105   89/105
 2019   F2 Table-1 mapping (Q05) 105         0.749           0.490    0.060    0.038      0.909  105/105   95/105
 2019    F3 thin-cell guard (R0) 100         0.733           0.563   -0.022    0.076      0.723        -        -
 2020     R0 rebuilt lab mapping 105    

**Takeaway (E):** F1 and F2 beat R0 on both M1 and M2 in 2019 and in every year 2020–2022, though F2's 2022 M1 gain rounds to 0.000 (F1 2019: M1 0.689 → 0.726, M2 0.453 → 0.487; F2: 0.749, 0.490). Both send Nigeria to the bottom: 39/105 under F1 (105/105 under F2, which also moves ch. 83 to Metals). F1 sends Congo 41 → 89. F3 drops 5 countries (AGO, BWA, COD, COG, NGA), because all their RCA>1 cells are thin, so the variant cannot score the countries the question is about. On its own sample it loses M1 in 2019, 2020 and 2021 (−0.022, −0.005, −0.062).

In [10]:
ANOM19 = sorted(SC.query('year == 2019 and anomalous').iso3)
rows = []
for excl_name, excl in [('none', []), ('NGA, COG', FOCUS), ('all 2019 anomalous: ' + ', '.join(ANOM19), ANOM19)]:
    for yy in CHK:
        for v in VN[1:3]:
            c = [i for i in CS[yy, v] if i not in excl]; e, e0 = VAR[yy, v][c], VAR[yy, VN[0]][c]
            rows.append({'excluded': excl_name, 'year': yy, 'variant': v, 'n': len(c),
                         'M1 diff': stats.spearmanr(e, ATL[yy][c])[0] - stats.spearmanr(e0, ATL[yy][c])[0],
                         'M2 diff': stats.spearmanr(e, LNY[yy][c])[0] - stats.spearmanr(e0, LNY[yy][c])[0]})
RB = pd.DataFrame(rows)
print(RB.pivot_table(index=['excluded', 'variant'], columns='year', values=['M1 diff', 'M2 diff']).round(3).to_string())
# which countries move most under F1 (2019)?
c = CS[2019, VN[1]]; r0, r1 = rank(VAR[2019, VN[0]], c), rank(VAR[2019, VN[1]], c)
mv = (r1 - r0).sort_values()
raw = R.raw_hs6(2019)
print('\nF1 2019 largest movers (rank change; negative = up):')
print(pd.DataFrame({'R0': r0[mv.index], 'F1': r1[mv.index], 'change': mv}).iloc[np.r_[0:6, -6:0]].astype(int).to_string())
print('countries whose rank moves by >= 10 under F1 (2019):', int((mv.abs() >= 10).sum()), 'of', len(c))

                                                                            M1 diff                      M2 diff                     
year                                                                           2019   2020   2021   2022    2019   2020   2021   2022
excluded                                         variant                                                                             
NGA, COG                                         F1 ch.89 -> Transportation   0.003 -0.008 -0.006  0.001   0.009 -0.015 -0.004  0.006
                                                 F2 Table-1 mapping (Q05)    -0.001 -0.008  0.006  0.000  -0.004  0.021  0.018  0.016
all 2019 anomalous: COD, COG, MDA, NGA, SAU, ZMB F1 ch.89 -> Transportation   0.001 -0.011 -0.005 -0.002   0.007 -0.018 -0.004  0.005
                                                 F2 Table-1 mapping (Q05)     0.000 -0.002  0.010 -0.003  -0.006  0.026  0.021  0.015
none                                             F1 ch.89 -> T

**Takeaway (E4, post hoc):** **almost all of F1's gain comes from Nigeria and Congo.** Excluding them, F1's ΔM1 is +0.003 / −0.008 / −0.006 / +0.001 and its ΔM2 is +0.009 / −0.015 / −0.004 / +0.006 (2019–2022), i.e. roughly zero. F2 keeps a small income gain without them in 2020–2022 (ΔM2 +0.021 / +0.018 / +0.016), but its ΔM1 is ≈0 (−0.001 / −0.008 / +0.006 / 0.000). F1 is low-risk: only **4 of 105** countries move ≥10 ranks in 2019 (Nigeria 18 → 39, Congo 41 → 89, Croatia 39 → 49 down; Poland 33 → 23 up). Korea and Italy fall 6 places each.

In [11]:
rng = np.random.default_rng(20261124); yy = 2019; B = 2000
boot = {}
for v in VN[1:]:
    c = np.array(CS[yy, v]); at, ly = ATL[yy][c].to_numpy(), LNY[yy][c].to_numpy()
    e0, e1 = VAR[yy, VN[0]][c].to_numpy(), VAR[yy, v][c].to_numpy()
    out = []
    for i in rng.integers(0, len(c), size=(B, len(c))):
        out.append((stats.spearmanr(e1[i], at[i])[0] - stats.spearmanr(e0[i], at[i])[0], stats.spearmanr(e1[i], ly[i])[0] - stats.spearmanr(e0[i], ly[i])[0]))
    boot[v] = out
dec = []
for v in VN[1:]:
    s = P[P.variant == v].set_index('year'); bb = np.array(boot[v])
    q1, q2 = np.percentile(bb[:, 0], [2.5, 97.5]), np.percentile(bb[:, 1], [2.5, 97.5])
    both = (s['M1 diff'] > 0) & (s['M2 diff'] > 0)
    dec.append({'variant': v, 'M1 diff 2019': round(s.loc[2019, 'M1 diff'], 3), 'M1 95% CI': f'[{q1[0]:.3f}, {q1[1]:.3f}]',
                'M2 diff 2019': round(s.loc[2019, 'M2 diff'], 3), 'M2 95% CI': f'[{q2[0]:.3f}, {q2[1]:.3f}]',
                'M1 wins 2019-22': int((s.loc[CHK, 'M1 diff'] > 0).sum()), 'M2 wins 2019-22': int((s.loc[CHK, 'M2 diff'] > 0).sum()),
                'both 2019': bool(both[2019]), 'both, of 2020-22': int(both.loc[[2020, 2021, 2022]].sum()),
                'BETTER (rule A)': bool(both[2019] and both.loc[[2020, 2021, 2022]].sum() >= 2)})
print(pd.DataFrame(dec).to_string(index=False))

                   variant  M1 diff 2019       M1 95% CI  M2 diff 2019       M2 95% CI  M1 wins 2019-22  M2 wins 2019-22  both 2019  both, of 2020-22  BETTER (rule A)
F1 ch.89 -> Transportation         0.037 [-0.002, 0.095]         0.034  [0.004, 0.077]                4                4       True                 3             True
  F2 Table-1 mapping (Q05)         0.060 [-0.018, 0.168]         0.038 [-0.037, 0.120]                4                4       True                 3             True
   F3 thin-cell guard (R0)        -0.022 [-0.116, 0.072]         0.076 [-0.070, 0.238]                1                2      False                 1            False


**Takeaway (E2):** **F1 and F2 pass rule A; F3 doesn't.** F1 is the first variant to pass rule A since the rule was introduced (11-19 to 11-21 scored 0 passes). The 2019 bootstrap is only moderately strong: F1's M2 gain excludes 0 ([0.004, 0.077]) and its M1 gain barely doesn't ([−0.002, 0.095]). F2's intervals both include 0. With E4 in mind, the honest reading is: *F1 and F2 fix a real misclassification that distorts two to three countries a year, and they are roughly neutral for everyone else.*

In [12]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.6), sharex=True)
COL = {'F1 ch.89 -> Transportation': '#2a78d6', 'F2 Table-1 mapping (Q05)': '#1baf7a', 'F3 thin-cell guard (R0)': '#eb6834'}
for ax, m, t in zip(axes, ['M1 diff', 'M2 diff'], ['ΔM1 vs R0 (Spearman with ATLAS ECI)', 'ΔM2 vs R0 (Spearman with log GDPpc)']):
    for v in VN[1:]:
        s = P[P.variant == v]; ax.plot(s.year, s[m], color=COL[v], lw=2, marker='o', ms=5, label=v)
    ax.axhline(0, color='#bdbcb8', lw=0.8); ax.axvspan(2022.5, 2023.5, color='#efefec', zorder=0)
    ax.set_title(t, fontsize=10, loc='left'); ax.set_xticks(YEARS); ax.grid(axis='y', color='#ececea', lw=0.8)
    for s_ in ['top', 'right']: ax.spines[s_].set_visible(False)
axes[0].set_ylabel('Δ Spearman ρ'); axes[1].legend(frameon=False, fontsize=8)
fig.text(0.99, 0.01, 'shaded: 2023 (suspect, D1)', ha='right', fontsize=7.5, color='#52514e')
fig.tight_layout(rect=(0, 0.03, 1, 1)); fig.savefig(FIG / 'delta_metrics.png', dpi=100); plt.close(fig)
print('saved figures/delta_metrics.png')

saved figures/delta_metrics.png


![delta metrics](figures/delta_metrics.png)

**Takeaway (figure):** F1 (blue) and F2 (green) stay at or above zero on both metrics in 2019–2022, with the largest gains in 2019, the year of Nigeria's one-off lines. F3 (orange) falls below zero on ATLAS agreement.

### F. Spring 2026 report countries

In [13]:
rows = []
for yy in [2019, 2022]:
    c = CS[yy, VN[0]]
    for iso, nm in R.REPORT_COUNTRIES.items():
        r = {'year': yy, 'country': nm, 'cleaned base': int(rank(GB[yy][0], c)[iso]) if iso in c else '-'}
        for v in VN[:3]:
            r[v.split()[0]] = int(rank(VAR[yy, v], c)[iso]) if iso in c else '-'
        c3 = CS[yy, VN[3]]
        r['F3 (own sample)'] = f"{int(rank(VAR[yy, VN[3]], c3)[iso])}/{len(c3)}" if iso in c3 else 'dropped'
        rows.append(r)
print(f'ranks within the joint common sample ∩ R0 (n = {len(CS[2019, VN[0]])} in 2019, {len(CS[2022, VN[0]])} in 2022); F3 in its own sample')
print(pd.DataFrame(rows).to_string(index=False))

ranks within the joint common sample ∩ R0 (n = 105 in 2019, 105 in 2022); F3 in its own sample
 year   country  cleaned base  R0  F1  F2 F3 (own sample)
 2019    Canada            95  82  79  62          72/100
 2019   Germany             1   1   1   6          11/100
 2019    Poland            34  33  23  36          41/100
 2019 Singapore            26  25  26  19           6/100
 2019       UAE           103 104 103 104           1/100
 2019        UK            37  35  34  12          31/100
 2019   Vietnam            32  31  32  34          43/100
 2022    Canada            73  73  74  61           65/92
 2022   Germany             7   7   7  12           11/92
 2022    Poland            38  38  33  32           43/92
 2022 Singapore             8   8   9   7            3/92
 2022       UAE           102 102 102 103            1/92
 2022        UK            12  12  12   5            8/92
 2022   Vietnam            34  34  37  38           35/92


**Takeaway (F):** F1 barely touches the report countries, except Poland (2019 rank 33 → 23; 2022 38 → 33). Poland's ch. 89 exports (ships, yachts) move into the top-PCI group. Germany, Singapore, the UAE and the UK move ≤3 places under F1 in both years, and Vietnam and Canada ≤3. F2 moves more (UK 35 → 12 and Canada 82 → 62 in 2019), as Q05 found. F3 would put the UAE at #1, which is disqualifying.

## Caveats

- **ATLAS is a benchmark, not ground truth.** But here the HS6 evidence (B, B3) shows the grouped ranks for Nigeria and Congo are driven by scrap vessels, platforms and one line of steel tubing. The diagnosis doesn't depend on ATLAS.
- **830710 for Nigeria may itself be a data error** (2019-only, $2.096B). We can't check it against mirror (importer) data without the Growth Lab country × HS4 file, which isn't in the repo (see data request). F1 doesn't touch it. F2 moves ch. 83 into Metals, which absorbs it.
- **Small samples and few anomalous countries** (2–6 per year). The Fisher tests are descriptive. The anomaly threshold (40 places) and thin threshold (50%) were pre-stated but are arbitrary.
- **E compares against R0 (rebuilt from raw), not the cleaned CSV.** They agree closely (Spearman ≥0.987), but 2019–2021 cleaned cells don't all equal the raw sums (Q05). The C counterfactuals use the cleaned CSV directly.
- **E4 is post hoc.** It changes how F1's rule-A pass should be read, not the pass itself.
- Nominal values; 2023 suspect (D1). Congo is absent from the 2022–2023 raw pull.

## Helpful to know

- **Grouped ECI = mean PCI of the RCA>1 groups, divided by σ₂** (exact; A). To explain any grouped rank, list the basket and the 18 PCIs. No other computation is needed.
- **HS ch. 89 (ships, boats, floating structures) is in *Daily Instruments*** in the lab mapping (`research.LAB_GROUPS`, from `datacleaning2025.ipynb` cell 5). Since Daily Instruments is the 3rd-most complex group (2019), any scrap-ship or oil-platform re-export lifts a country's ECI. ATLAS rates those products at about the 11th PCI percentile.
- **Ch. 99 (999999, unspecified) feeds both *Services and Utilities* and *High-complexity Products*.** So countries that report oil under 999999 (Saudi Arabia, $202B) get *two* high-PCI cells from it. 11-17 found the first effect. This run shows the aggregate double-count, which put Saudi Arabia **2/105** in 2019.
- **Thin cells are the norm, not the exception** (≈80% of countries have one). A blanket concentration rule removes commodity exporters' main cells and breaks the ECI (F3). Fixes should target specific misfiled chapters.
- Congo's (COG, M49 178) raw rows are missing in 2022–2023, so it drops out of every grouped analysis in those years.

## New research avenues (also in `proposals/2026-11-24.md`)

1. **Remove ch. 99 from the *High-complexity Products* aggregate** (one line in the mapping) and score it with M1/M2 and rule A, alone and combined with F1. Saudi Arabia (2/105 vs ATLAS 45) is the test case. Repo data, one night.
2. **Chapter-by-chapter mapping audit with ATLAS PCI.** For each HS2 chapter, compare the mean 2019 ATLAS PCI of its HS4 products with the PCI of the lab group it's mapped to. Flag chapters whose ATLAS PCI percentile and group PCI rank disagree by more than half the scale, the way ch. 89 does. This generalises tonight's finding into a mapping checklist for the next data update. One night.
3. **Commodity lines in "complex" groups (DR Congo, Zambia).** Score a variant that moves 2822 (cobalt) and 2807 (sulphuric acid) out of *Chemicals*, and unrefined/refined copper (7402/7403) into a resource group, using the same rule and E4-style exclusion check. One night.
4. **F1 for the greedy target (Q11-style, no greedy run).** For each Spring 2026 country, does F1 change which group gives the largest marginal ECI gain? Poland, whose rank moves 10 places, is the case to check. One night.

## Data issues / data needed

- **Nigeria 2019, HS6 830710 = $2.096B** (flexible tubing of iron or steel), present only in 2019. It may be a misreport. It can't be checked against importer-side data without the Growth Lab country × HS4 file (`external-data/raw/atlas_hs92_country_product_year_4.csv` / `atlas_hs4_exports.pkl`, gitignored and not present in this environment). Details: `data-requests/2026-11-24-nigeria-830710-mirror.md`.
- **Republic of Congo (M49 178) is absent from the 2022 and 2023 raw pulls.** This is noted for the 2024 re-pull and is in the same data request.

In [14]:
print(f'Run time: {time.time() - T0:.1f} s')

Run time: 13.0 s
